# Introduction

Project scope is using the MNIST digit dataset to train a 10 layer MLP on using Sigmoid and ReLU activation functions for my Machine Learning course on Deep Learning.

# Environment Setup

In [ ]:
import sys
assert sys.version_info >= (3, 7)

In [ ]:
from packaging import version
import sklearn
from sklearn.model_selection import train_test_split

assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

In [ ]:
import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

Having minor issues with kaggle throwing warnings from this block. Seems benign, but might need to fiddle with the version assert

In [ ]:
import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Running this directly on kaggle for easier importing of the dataset, would need to include an alternate if I intended for the notebook to be portable. Something to think about for later.

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

Just setting up visualization settings.

# Importing Data

In [ ]:
test = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")
train = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")

In [ ]:
train.info()
train.head()

In [ ]:
Y_training = train["label"]
X_training = train.drop(columns=["label"])


X_test = test
X_test.head()

In [ ]:
X_train, X_validation, Y_train, Y_validation = train_test_split(
    X_training, Y_training,
    test_size=0.2,
    random_state=42,
    stratify=Y_training
)

In [ ]:
Y_training.info()

In [ ]:
X_train.info()
X_train.head()

In [ ]:
X_train = X_train / 255
X_validation = X_validation / 255
X_test_scaled = X_test / 25

In [ ]:
plt.imshow(X_train.iloc[0].values.reshape(28, 28), cmap="binary")
plt.axis("off")
plt.show()

## Feature Engineering

Data is clean and with nothing missing, so I don't need to address any glaring issues. Looking through the data, it looked like I could maybe shave some of the pixels on the sides. I know if this was using more basic classifiers that it might be worth doing, but currently it seems fine.

# Model Creation

In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(72)

In [ ]:
optimizer  = tf.keras.optimizers.Adam(learning_rate=1e-3)

optimizer idea lifted from the textbook, gave far better results for model 1. Gave errors when I tried to use it for model 2 and 3, need to figure out why. Adam gave way better results.

## Model 1: Sigmoid Activation

In [ ]:
model_1 = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(100, activation="sigmoid"),
    tf.keras.layers.Dense(100, activation="sigmoid"),
    tf.keras.layers.Dense(100, activation="sigmoid"),
    tf.keras.layers.Dense(100, activation="sigmoid"),
    tf.keras.layers.Dense(100, activation="sigmoid"),
    tf.keras.layers.Dense(100, activation="sigmoid"),
    tf.keras.layers.Dense(100, activation="sigmoid"),
    tf.keras.layers.Dense(100, activation="sigmoid"),
    tf.keras.layers.Dense(100, activation="sigmoid"),
    tf.keras.layers.Dense(100, activation="sigmoid"),
    tf.keras.layers.Dense(10, activation="softmax")
    ])

In [ ]:
model_1.summary()

In [ ]:
##model_1.compile(loss="sparse_categorical_crossentropy",
  ##            optimizer = "sgd",
  ##            metrics=["accuracy"])

Original version commented out, using sgd as the optimizer gave abysmal accuracy, failing to hit even 20% after 10 epochs. 

In [ ]:
model_1.compile(loss="sparse_categorical_crossentropy",
              optimizer = optimizer,
              metrics=["accuracy"])

## Model 2: ReLU Activation

In [ ]:
model_2 = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
    ])

In [ ]:
model_2.summary()

In [ ]:
model_2.compile(loss="sparse_categorical_crossentropy",
              optimizer= "sgd",
              metrics=["accuracy"])

## Model 3: ReLU + Batch Normalization

In [ ]:
model_3 = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(10, activation="softmax")
    ])

In [ ]:
model_3.summary()

In [ ]:
model_3.compile(loss="sparse_categorical_crossentropy",
              optimizer= "sgd",
              metrics=["accuracy"])

In [ ]:
##tf.keras.utils.plot_model(model_3, show_shapes=True) ##Shows the details, but is huge due to the sheer number of layers

# Model Training

Models 1, 2 and 3 built, now to train them.

All three are getting 10 epochs, tried going for 20 but wound up seeing 100% accuracy on training data, so dialed it back.

##  Model 1 Training

In [ ]:
history_1 = model_1.fit(X_train, Y_train, epochs=10,
                    validation_data=(X_validation, Y_validation))



## Model 2 Training

In [ ]:
history_2 = model_2.fit(X_train, Y_train, epochs=10,
                    validation_data=(X_validation, Y_validation))

## Model 3 Training

In [ ]:
history_3 = model_3.fit(X_train, Y_train, epochs=10,
                    validation_data=(X_validation, Y_validation))

# Results

## Model Stats Summary

* Model 1 with Sigmoid activation: Training: .89 Accuracy, .42 loss. Validation: .89 accuracy, .43 loss. 62 seconds training time
* Model 2 with ReLU activation: Training: .97 Accuracy, .07 loss. Validation: .95 accuracy, .17 loss. 51 seconds training time
* Model 3 with ReLU and Batch Norm: Training: .99 Accuracy, .03 loss. Validation: .95, loss .17. 74 seconds training time

In [ ]:
h1 = history_1.history
h2 = history_2.history
h3 = history_3.history

epochs_1 = range(1, len(h1['loss']) + 1)
epochs_2 = range(1, len(h2['loss']) + 1)
epochs_3 = range(1, len(h3['loss']) + 1)

plt.figure(figsize=(16, 6))

# Plot Training & Validation Loss 
plt.subplot(1, 2, 1)
plt.plot(epochs_1, h1['loss'], 'b-', label='Model 1 Training Loss')
plt.plot(epochs_1, h1['val_loss'], 'b--', label='Model 1 Validation Loss')
plt.plot(epochs_2, h2['loss'], 'r-', label='Model 2 Training Loss')
plt.plot(epochs_2, h2['val_loss'], 'r--', label='Model 2 Validation Loss')
plt.plot(epochs_3, h3['loss'], 'g-', label='Model 3 Training Loss')
plt.plot(epochs_3, h3['val_loss'], 'g--', label='Model 3 Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Plot Training & Validation Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_1, h1['accuracy'], 'b-', label='Model 1 Training Accuracy')
plt.plot(epochs_1, h1['val_accuracy'], 'b--', label='Model 1 Validation Accuracy')
plt.plot(epochs_2, h2['accuracy'], 'r-', label='Model 2 Training Accuracy')
plt.plot(epochs_2, h2['val_accuracy'], 'r--', label='Model 2 Validation Accuracy')
plt.plot(epochs_3, h3['accuracy'], 'g-', label='Model 3 Training Accuracy')
plt.plot(epochs_3, h3['val_accuracy'], 'g--', label='Model 3 Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

# Reflection

First, code blocks to generate a prediction csv file for submission.

In [ ]:
y_pred_proba = model_2.predict(X_test_scaled)
y_pred = np.argmax(y_pred_proba, axis=1)

In [ ]:
##y_pred_proba = model_3.predict(X_test_scaled)
##y_pred = np.argmax(y_pred_proba, axis=1)

Model 3 results looked great, but on submission scored rather low. Not entirely sure of why, but batch norm is the only difference code-wise.

In [ ]:
submission = pd.DataFrame({
    "ImageId": np.arange(1, len(y_pred)+1),
    "Label": y_pred
})
submission.to_csv("submission.csv", index=False)

Can definitely see the advantage of using Batch norm, it performed far better after even the first epoch and it took ReLU alone almost four epochs to catch up. Need to research why it then underperformed for prediction, as it should have done far better.